# EyeStim: auditoría de adecuación de BioID

Objetivo: comprobar si las variables y anotaciones de BioID permiten validar localización pupilar, diámetro, mirada y atención. La fuente oficial describe BioID como un benchmark de detección facial con posiciones manuales de ambos ojos, no como un estudio de pupilometría o atención: https://www.bioid.com/face-database/

In [1]:
from pathlib import Path
import sys
import torch

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))
from research.validation import audit_bioid
from dataset import EyeStimDataset
from torch.utils.data import random_split

BIOID = ROOT / 'data' / 'bioid'
assert BIOID.exists(), 'Ejecuta python src/dataset.py para descargar BioID.'

In [2]:
integrity, fitness = audit_bioid(BIOID)
display(integrity)
display(fitness)
assert integrity.loc[integrity.metric == 'paired_samples', 'value'].iloc[0] == 1521
assert (fitness.loc[fitness.objetivo.isin(['Elipse/diámetro pupilar', 'Dirección o punto de mirada', 'Atención/carga cognitiva']), 'cubierto'] == 'No').all()

,metric,value
0,images,1521
1,annotation_files,1521
2,paired_samples,1521
3,missing_annotations,0
4,missing_images,0
5,malformed_or_unreadable,0
6,out_of_bounds_annotations,0
7,unique_image_shapes,"[(286, 384)]"
8,median_inter_eye_distance_px,52.038447


,objetivo,cubierto,evidencia
0,Detección/localización facial,Sí,Rostros y centros aproximados de ambos ojos
1,Centro pupilar preciso,No,"La etiqueta representa posición del ojo, no ce..."
2,Elipse/diámetro pupilar,No,"No hay máscara, ejes ni diámetro de pupila"
3,Dirección o punto de mirada,No,No hay objetivo de pantalla ni vector de mirada
4,Atención/carga cognitiva,No,"No hay tarea, condición, conducta ni etiqueta ..."
5,Partición por participante,No con el loader actual,Los nombres usados no codifican el sujeto


## Prueba de la partición actual

`random_split` crea dos vistas del mismo objeto. Cambiar `val_dataset.dataset.train` también modifica el conjunto de entrenamiento. Sin *jitter*, el recorte se centra en la etiqueta y todos los objetivos normalizados son `(0, 0)`.

In [3]:
dataset = EyeStimDataset(data_dir=str(BIOID), train=True)
val_size = int(len(dataset) * 0.2)
train_subset, val_subset = random_split(dataset, [len(dataset) - val_size, val_size], generator=torch.Generator().manual_seed(42))
same_object = train_subset.dataset is val_subset.dataset
val_subset.dataset.train = False
targets = {tuple(dataset[i][1].tolist()) for i in range(100)}
print({'same_dataset_object': same_object, 'train_flag_after_validation_change': dataset.train, 'unique_targets_first_100': targets})
assert same_object and dataset.train is False and targets == {(0.0, 0.0)}

{'same_dataset_object': True, 'train_flag_after_validation_change': False, 'unique_targets_first_100': {(0.0, 0.0)}}


## Dictamen

BioID tiene integridad interna para su propósito original, pero **no es adecuado como dataset principal de EyeStim**. Puede conservarse únicamente como auxiliar para detección facial/ocular. No permite estimar ni validar diámetro pupilar, mirada a pantalla o atención; además, la evaluación actual es tautológica por el centrado con la etiqueta.